# M3L3 E14 — Support bot multiagente con LangGraph
### Módulo 3 · Lecture 3 · Sistemas Multiagente

**Ejercicio paralelo:** E11 (support bot multiagente en Python puro)

## ¿Qué vas a aprender hoy?
- trasladar la arquitectura de E11 a un `StateGraph` con LLM real.
- convertir el `AgentResponse` TypedDict de E11 en campos del State.
- inspeccionar el grafo del bot completo con `draw_mermaid()`.


## ¿Qué necesitás saber antes?

Venís de E11 donde construiste el support bot con `AgentResponse`, `detect_domains` y `handle_query` en Python puro. En E14 esa misma arquitectura se declara como grafo y los agentes usan el LLM real.

> **Traslado a LangGraph:** convertir un sistema Python puro a LangGraph significa mapear cada función del orquestador a un nodo, y cada decisión de flujo a un edge.

| E11 Python puro | E14 LangGraph |
|---|---|
| `AgentResponse` TypedDict aparte | Campos del `SupportState` |
| `detect_domains(query)` con keywords | LLM clasifica el dominio |
| `specialist_agent(domain, query)` responde con KB lookup | LLM genera respuesta usando contexto de KB |
| `handle_query(query)` orquesta todo | `graph.compile()` + `app.invoke()` |
| Sin visualización del flujo | `draw_mermaid()` genera el diagrama |


## Paso 1 — Elegí tu proveedor de LLM

In [ ]:
PROVIDER = "openai"   # ← cambiá esto: "openai" | "gemini" | "claude"

import os
from getpass import getpass

if PROVIDER == "openai":
    !pip install langchain-openai -q
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

elif PROVIDER == "gemini":
    !pip install langchain-google-genai -q
    from langchain_google_genai import ChatGoogleGenerativeAI
    os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ")
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

elif PROVIDER == "claude":
    !pip install langchain-anthropic -q
    from langchain_anthropic import ChatAnthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

else:
    raise ValueError(f"PROVIDER inválido: {PROVIDER!r}. Opciones: 'openai' | 'gemini' | 'claude'")

print(f"LLM listo → proveedor: {PROVIDER}")

In [ ]:
!pip install langgraph -q

from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

print("LangGraph listo.")

## Sección 1 — Del AgentResponse al State

> **Integración del protocolo:** en E11 el `AgentResponse` era un objeto separado. En LangGraph, esos campos viven directamente en el State, accesibles para cualquier nodo.

```python
# E11
class AgentResponse(TypedDict):
    agent_name: str
    status: Literal["success", "out_of_scope"]
    answer: str
    confidence: Literal["high", "medium", "low"]
    sources: list[str]
```

Ahora esos campos forman parte del `SupportState`, junto con `query` y `domain`.

In [ ]:
knowledge_base = {
    "hr": [
        "Vacaciones: cada empleado tiene 15 días hábiles por año.",
        "Beneficios: el seguro médico inicia el primer día de trabajo.",
        "Licencias: registrar el pedido en PeopleOps y avisar al manager.",
    ],
    "tech": [
        "VPN: reiniciar el cliente, validar MFA y abrir ticket si persiste.",
        "Contraseña: restablecer desde el portal de identidad.",
        "Notebook: reportar equipo dañado con número de serie.",
    ],
    "billing": [
        "Facturas: cargar comprobantes antes del día 25.",
        "Reembolsos: adjuntar recibo, monto y centro de costo.",
        "Pagos: se procesan los viernes por la tarde.",
    ],
}

In [ ]:
class SupportState(TypedDict):
    query: str
    domain: str
    agent_name: str
    status: Literal["success", "out_of_scope"]
    answer: str
    confidence: Literal["high", "medium", "low"]
    sources: list

## Sección 2 — Nodos

> **Nodo especialista:** usa el LLM con contexto de la KB para generar una respuesta. Devuelve los campos del AgentResponse.

El nodo `detect` usa el LLM para clasificar el dominio. Los nodos especialistas ya están implementados con `_specialist_response`. Tu tarea es implementar `unknown_node` y `domain_router`.

```
SupportState.domain = "hr"     → domain_router → "hr_node"
SupportState.domain = "tech"   → domain_router → "tech_node"
SupportState.domain = "billing"→ domain_router → "billing_node"
SupportState.domain = "unknown"→ domain_router → "unknown_node"
```

In [ ]:
def detect(state: SupportState) -> dict:
    prompt = (
        "Clasificá esta consulta de soporte corporativo en exactamente una categoría.\n"
        "Categorías: hr, tech, billing, unknown\n"
        "Respondé SOLO con la categoría.\n\n"
        f"Consulta: {state['query']}"
    )
    response = llm.invoke(prompt)
    domain = response.content.strip().lower()
    if domain not in ("hr", "tech", "billing"):
        domain = "unknown"
    return {"domain": domain}


def _specialist_response(domain: str, display: str, state: SupportState) -> dict:
    context = "\n".join(knowledge_base[domain])
    response = llm.invoke(
        f"Sos el agente {display} de soporte corporativo.\n"
        f"Usando solo este contexto:\n{context}\n\n"
        f"Respondé brevemente en español: {state['query']}"
    )
    return {
        "agent_name": display,
        "status": "success",
        "answer": response.content.strip(),
        "confidence": "high",
        "sources": knowledge_base[domain],
    }


def hr_node(state: SupportState) -> dict:
    return _specialist_response("hr", "HRAgent", state)


def tech_node(state: SupportState) -> dict:
    return _specialist_response("tech", "TechAgent", state)


def billing_node(state: SupportState) -> dict:
    return _specialist_response("billing", "BillingAgent", state)


def unknown_node(state: SupportState) -> dict:
    # TODO: devolver agent_name="UnknownAgent", status="out_of_scope",
    # answer="Consulta fuera del alcance del sistema.", confidence="low", sources=[]
    return {}


def domain_router(state: SupportState) -> str:
    # TODO: mapear state["domain"] → nombre del nodo
    # hr → "hr_node", tech → "tech_node", billing → "billing_node", * → "unknown_node"
    return "unknown_node"

## Sección 3 — Construir el grafo del support bot

```
START → detect → hr_node      → END
               → tech_node    → END
               → billing_node → END
               → unknown_node → END
```

`draw_mermaid()` va a mostrar ese diagrama exacto después de compilar.

In [ ]:
graph = StateGraph(SupportState)

graph.add_node("detect",       detect)
graph.add_node("hr_node",      hr_node)
graph.add_node("tech_node",    tech_node)
graph.add_node("billing_node", billing_node)
graph.add_node("unknown_node", unknown_node)

graph.add_edge(START, "detect")
graph.add_conditional_edges(
    "detect", domain_router,
    {"hr_node": "hr_node", "tech_node": "tech_node",
     "billing_node": "billing_node", "unknown_node": "unknown_node"},
)
for node in ["hr_node", "tech_node", "billing_node", "unknown_node"]:
    graph.add_edge(node, END)

app = graph.compile()
print("Grafo compilado.")

In [ ]:
EMPTY = {"query": "", "domain": "", "agent_name": "", "status": "success", "answer": "", "confidence": "low", "sources": []}

for q in ["¿Cuántos días de vacaciones tengo?", "No funciona la VPN", "Necesito cargar una factura", "¿Dónde almuerzo?"]:
    r = app.invoke({**EMPTY, "query": q})
    print(f"[{r['agent_name']:12s}][{r['status']:12s}] {r['answer'][:55]}")

In [ ]:
print(app.get_graph().draw_mermaid())

## Checks automáticos

In [ ]:
def run_checks():
    r1 = app.invoke({**EMPTY, "query": "quiero pedir vacaciones"})
    assert r1["agent_name"] == "HRAgent" and r1["status"] == "success"

    r2 = app.invoke({**EMPTY, "query": "no puedo conectarme a la VPN"})
    assert r2["agent_name"] == "TechAgent"

    r3 = app.invoke({**EMPTY, "query": "¿qué hace el departamento de arte?"})
    assert r3["status"] == "out_of_scope", f"esperaba out_of_scope: {r3['status']}"
    assert r3["agent_name"] == "UnknownAgent"

    print("Checks E14 OK")

run_checks()

## ¿Qué aprendiste hoy?

- El LLM en `detect` entiende semánticamente el dominio: no necesita keywords exactas.
- Los nodos especialistas usan RAG + LLM: el contexto de la KB guía la respuesta generada.
- `draw_mermaid()` produce automáticamente la documentación visual del bot completo.

## Próximo ejercicio

En **E15** vas a convertir los agentes en `@tool` y usar `bind_tools()` para que el LLM seleccione tools de forma nativa.
